In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# Heapsort

## Graphical Representation

In [ ]:
import { Graphviz } from "@hpcc-js/wasm";
import { display } from "tslab";

The function `toDot` takes four arguments:
- `A` is an array of natural numbers of length $n$,
- `f` is a natural number such that $0 \leq f < n$ holds,
- `g` is a natural number such that $f < g < n$ holds,
- `u` is a natural number such that $0 \leq u < n$ holds.
  This argument is optional.

The function returns a graphical representation of the array `A` as a heap. 
This graphical representation is stored as a directed graph with an encoding suitable for `Graphviz`. 

The part `A[0:g]` is represented as a binary tree, while the part `A[g:]` is represented as an array.  Furthermore, all indexes in the range `A[k:g]` satisfy the heap condition.  The nodes in the range `[0:k-1]` are colored red.  If `u` is set, the node `A[u]` is colored orange.

In [ ]:
function toDot(A: number[], f: number, g: number, u?: number): string {
  const n = A.length;
  const parts: string[] = [];
  parts.push("digraph G {");
  parts.push('  node [shape=record];');

  for (let k = 0; k < n; k++) {
    const label = `{ ${A[k]} | ${k} }`;
    let style = "";
    if (u !== undefined && k === u) style = 'style=filled, fillcolor=orange';
    else if (k < f) style = 'style=filled, fillcolor=red';
    else if (k < g) style = 'style=rounded';
    else style = 'style=filled, fillcolor=green';
    parts.push(`  "${k}" [label="${label}", ${style}];`);
  }

  for (let k = 0; k <= Math.floor((n - 2) / 2); k++) {
    const left = 2 * k + 1;
    const right = 2 * k + 2;
    if (left < g) parts.push(`  "${k}" -> "${left}";`);
    if (right < g) parts.push(`  "${k}" -> "${right}";`);
  }

  parts.push("}");
  return parts.join("\n");
}

const gv = await Graphviz.load();

function showGraph(A: number[], f: number, g: number, u?: number) {
  const dot = toDot(A, f, g, u);
  const svg = gv.layout(dot, "svg", "dot");
  display.html(svg);
}


# HeapSort

The function call `swap(A, i, j)` takes an array `A` and  two indexes `i` and `j` and exchanges the elements at these indexes.

In [ ]:
function swap(A: number[], i: number, j: number): void {
  const t = A[i];
  A[i] = A[j];
  A[j] = t;
}

The function `ascend` takes two arguments:
- `A` is an array.
- `k` is an index into the array `A`.

   Therefore we have $k \in \bigl\{0, \cdots, \texttt{len}(A)-1\bigr\}$.

The array `A` represents a *heap*.  However, the <em style="color:blue">heap condition</em> might be violated 
at index `k`: It might be the case that the element at this index is to small and needs to rise to the top
of the heap.  The function `ascend` will fix the heap condition and will rise the element `A[k]` as much 
as is necessary to turn `A` into a heap.

In [ ]:
function ascend(A: number[], k: number): void {
  while (k > 0) {
    const p = Math.floor((k - 1) / 2);
    if (A[k] < A[p]) {
      swap(A, p, k);
      k = p;
    } else {
      return;
    }
  }
} 

The function `sink` takes three arguments.
- `A` is the array representing the heap.
- `k` is an index into the array `A`.
- `n` is the upper bound  of the part of this array that has to be transformed into a heap.  

The array `A` itself might actually have more than $n+1$ elements, but for the
purpose of the method `sink` we restrict our attention to the subarray
`A[k:n]`. 
When calling `sink`, the assumption is that `A[k:n+1]` should represent a heap 
that possibly has its heap condition violated at its root, i.e. at index `k`.  The
purpose of the procedure `sink` is to restore the heap condition at index `k`.
- We compute the index `j` of the left subtree below index `k`.
- We check whether there also is a right subtree at position `j+1`.
  
  This is the case if `j + 1 <= n`.  
- If the heap condition is violated at index `k`, we exchange the element at  position `k` 
  with the child that has the higher priority, i.e. the child that is smaller. 
- Next, we check in line 9 whether the heap condition is violated at index `k`.  
  If the heap condition is satisfied, there is nothing left to do and the procedure returns.  
  
- Otherwise, the element at position `k` is swapped with
  the element at position `j`.  
  
  Of course, after this swap it is possible that the heap condition is
  violated at position `j`.  Therefore,  `k` is set to `j` and the `while`-loop continues
  as long as the node at position `k` has at least one child, i.e. as long as 
  `2 * k + 1 <= n`.

In [ ]:
function descend(A: number[], k: number, n: number): void {
  while (2 * k + 1 <= n) {
    let j = 2 * k + 1;
    if (j + 1 <= n && A[j] > A[j + 1]) j += 1;
    if (A[k] <= A[j]) return;
    swap(A, k, j);
    k = j;
  }
}

The function $\texttt{heapSort}$ has the task to sort the array `A` and proceeds in two phases.
- In phase one our goal is to transform the array `A`into a heap that is stored in `A`.

  In order to do so, we traverse the array `A` in reverse in a loop.  
  The invariant of this loop is that before `ascend` is called, the array `A[:k]`
  is a heap.  The call `ascend(A, k)` inserts the element `A[k]` at the proper place and
  thereby extends the heap to `A[:k+1]`.
- In phase two we remove the elements from the heap one-by-one and insert them at the end of
  the array.

  When the `while`-loop starts, the array `A` contains a heap.  Therefore,
  the smallest element is found at the root of the heap.  Since we want to sort the
  array `A` descendingly, we move this element to the end of the array `A` and in
  return move the element from the end of the array`A`to the front.
  After this exchange, the subarray `A[0:n-1]` represents a heap, except that the
  heap condition might now be violated at the root.  Next, we decrement `n`, since the
  last element of the array `A` is already in its correct position.  
  In order to reestablish the heap condition at the root, we call `sink` with index `0`.

In [ ]:
function heapSort(A: number[]): void {
  let n = A.length;

  for (let k = 1; k < n; k++) {
    showGraph(A, 0, k, k);
    ascend(A, k);
    showGraph(A, 0, k + 1);
  }

  n -= 1;

  while (n >= 1) {
    swap(A, 0, n);
    showGraph(A, 1, n + 1);
    n -= 1;
    descend(A, 0, n);
    showGraph(A, 0, n + 1);
  }
}

## Testing

In [ ]:
function randomInt(minIncl: number, maxIncl: number): number {
  return Math.floor(Math.random() * (maxIncl - minIncl + 1)) + minIncl;
}

In [ ]:

function demo() {
  const L = Array.from({ length: 12 }, () => randomInt(1, 200));
  console.log("L =", L.slice());
  heapSort(L);
  console.log("L =", L.slice());
}

In [ ]:
demo();